In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Monthly SWI grid
swi = pd.read_csv(
    "/content/sample_data/SWI_1990_2025.csv"
)

# Municipality-grid mapping
mapping = pd.read_parquet(
    "/content/sample_data/mapping_SWI_communes_weighted.parquet"
)

# CatNat database
df_catnat = pd.read_excel(
    "/content/sample_data/CatNat_secheresse_couvertes_clean.xlsx"
)

print(swi.shape)
print(mapping.shape)
print(df_catnat.shape)

(3879792, 5)
(88670, 3)
(74317, 7)


In [ ]:
print(swi["DATE"].min(), swi["DATE"].max(), swi["DATE"].nunique())

199001 202512 432


In [ ]:
print(swi.columns.tolist())
print(mapping.columns.tolist())
print(df_catnat.columns.tolist())

['NUMERO', 'LAMBX', 'LAMBY', 'DATE', 'SWI_UNIF_MENS']
['code_insee', 'num_maille', 'weight']
['code_insee', 'Nom_commune', "Début d'Evénement", "Fin d'Evénement", 'Nom du péril', 'decision', 'nb_decisions']


In [ ]:
# ==========================================================
# 1. Harmonize variable names
# ==========================================================

swi = swi.rename(columns={
    "NUMERO": "grid_id",
    "SWI_UNIF_MENS": "swi"
})

# Data types
swi["grid_id"] = swi["grid_id"].astype(int)

mapping = mapping.rename(columns={
    "num_maille": "grid_id"
})

mapping["grid_id"] = mapping["grid_id"].astype(int)
mapping["code_insee"] = (
    mapping["code_insee"]
    .astype(str)
    .str.zfill(5)
)

In [ ]:
# ==========================================================
# 2. Check dates
# ==========================================================

print(swi["DATE"].min())
print(swi["DATE"].max())

swi.head()

199001
202512


,grid_id,LAMBX,LAMBY,DATE,swi
0,2,641374,7106309,199001,0.673
1,2,641374,7106309,199002,0.907
2,2,641374,7106309,199003,0.846
3,2,641374,7106309,199004,0.736
4,2,641374,7106309,199005,0.489


In [ ]:
swi = swi.rename(columns={
    "NUMERO": "grid_id",
    "DATE": "month",
    "SWI_UNIF_MENS": "swi"
})

In [ ]:
# Anomalie standardisée par maille et par mois calendaire (référence 1991-2020)
swi["month"] = swi["month"].astype(int)
swi["cal_month"] = swi["month"] % 100
swi["year"] = swi["month"] // 100

ref = swi[(swi["year"] >= 1991) & (swi["year"] <= 2020)]
stats = (ref.groupby(["grid_id", "cal_month"])["swi"]
         .agg(swi_mean="mean", swi_std="std").reset_index())

swi = swi.merge(stats, on=["grid_id", "cal_month"], how="left")
swi["swi_z"] = (swi["swi"] - swi["swi_mean"]) / swi["swi_std"]
swi["swi_z"] = swi["swi_z"].fillna(0)       # série constante : anomalie nulle

print(swi["swi_z"].describe())
swi.head()

count    3.879792e+06
mean    -2.928463e-02
std      1.004202e+00
min     -9.498347e+00
25%     -7.435119e-01
50%     -3.827350e-02
75%      6.883188e-01
max      6.326900e+00
Name: swi_z, dtype: float64


,grid_id,LAMBX,LAMBY,month,swi,cal_month,year,swi_mean,swi_std,swi_z
0,2,641374,7106309,199001,0.673,1,1990,0.857000,0.144064,-1.277209
1,2,641374,7106309,199002,0.907,2,1990,0.878333,0.104644,0.273946
2,2,641374,7106309,199003,0.846,3,1990,0.857300,0.080724,-0.139984
3,2,641374,7106309,199004,0.736,4,1990,0.742367,0.096763,-0.065797
4,2,641374,7106309,199005,0.489,5,1990,0.598667,0.117886,-0.930279


In [ ]:
municipality_swi = swi[["grid_id", "month", "swi", "swi_z"]].merge(
    mapping, on="grid_id", how="inner"
)
print(municipality_swi.shape)

(38305440, 6)


In [ ]:
print((swi["swi_z"].abs() > 5).sum(), "valeurs avec |z| > 5")
print(swi.loc[swi["swi_z"].abs() > 5, "year"].value_counts().head(8))
print(swi.loc[swi["swi_z"].abs() > 5, "swi_std"].describe().round(3))

492 valeurs avec |z| > 5
year
1990    358
2024    118
2021      6
2022      5
2023      5
Name: count, dtype: int64
count    492.000
mean       0.083
std        0.017
min        0.042
25%        0.070
50%        0.079
75%        0.091
max        0.140
Name: swi_std, dtype: float64


In [ ]:
municipality_swi["w_swi"] = municipality_swi["swi"]   * municipality_swi["weight"]
municipality_swi["w_z"]   = municipality_swi["swi_z"] * municipality_swi["weight"]

monthly_swi = (
    municipality_swi
    .groupby(["code_insee", "month"], as_index=False)
    .agg(swi=("w_swi", "sum"), swi_z=("w_z", "sum"))
)

print(monthly_swi.shape)
print(monthly_swi["code_insee"].nunique())
monthly_swi.head()

(15008112, 4)
34741


,code_insee,month,swi,swi_z
0,01001,199001,0.643011,-4.716342
1,01001,199002,0.931039,-0.841393
2,01001,199003,0.868469,-0.827883
3,01001,199004,0.887110,0.409746
4,01001,199005,0.675408,-0.461743


In [ ]:
dfc = df_catnat.copy()

dfc["code_insee"] = dfc["code_insee"].astype(str).str.zfill(5)

dfc["start_month"] = pd.to_datetime(dfc["Début d'Evénement"]).values.astype("datetime64[M]")
dfc["end_month"]   = pd.to_datetime(dfc["Fin d'Evénement"]).values.astype("datetime64[M]")

dfc = dfc.reset_index(drop=True)
dfc["event_id"] = np.arange(len(dfc))

print(dfc.shape)

(74317, 10)


In [ ]:
def months_between(row):
    if pd.isna(row.start_month) or pd.isna(row.end_month):
        return []
    return pd.date_range(row.start_month, row.end_month, freq="MS")

expanded = (
    dfc[["event_id", "code_insee", "start_month", "end_month"]]
    .assign(month=dfc.apply(months_between, axis=1))
    .explode("month", ignore_index=True)
    .dropna(subset=["month"])
)

print(expanded.shape)

(729282, 5)


In [ ]:
monthly_swi["month"] = pd.to_datetime(monthly_swi["month"].astype(str), format="%Y%m")

joined = expanded.merge(monthly_swi, on=["code_insee", "month"], how="left")
print(joined.shape)
joined.head()

(729282, 7)


,event_id,code_insee,start_month,end_month,month,swi,swi_z
0,0,01004,2003-07-01,2003-09-01,2003-07-01,0.179560,-1.566030
1,0,01004,2003-07-01,2003-09-01,2003-08-01,0.065850,-1.648449
2,0,01004,2003-07-01,2003-09-01,2003-09-01,0.172641,-1.239062
3,1,01004,2008-04-01,2008-12-01,2008-04-01,1.076331,1.246608
4,1,01004,2008-04-01,2008-12-01,2008-05-01,0.868833,0.161374


In [ ]:
idx  = joined.groupby("event_id")["swi"].idxmin().dropna().astype(int)
idxz = joined.groupby("event_id")["swi_z"].idxmin().dropna().astype(int)

swi_event = (
    joined.loc[idx, ["event_id", "swi"]]
    .rename(columns={"swi": "swi_min"}).set_index("event_id")
    .join(
        joined.loc[idxz, ["event_id", "swi_z"]]
        .rename(columns={"swi_z": "swi_anom_min"}).set_index("event_id")
    )
)

print(swi_event.shape)
print(swi_event.describe())

(74076, 2)
            swi_min  swi_anom_min
count  74076.000000  74076.000000
mean       0.074450     -1.449189
std        0.162101      0.665563
min       -0.047764     -6.163972
25%       -0.005000     -1.769804
50%        0.014000     -1.380492
75%        0.073093     -1.044293
max        1.077539      2.805269


In [ ]:
final = (
    dfc.set_index("event_id")
       .join(swi_event, how="left")
       .reset_index(drop=True)
)

print(final.shape)
print(final[["swi_min", "swi_anom_min"]].describe())

final.to_excel("/content/CatNat_drought_events_SWI_anom_1991_2020.xlsx", index=False)

(74317, 11)
            swi_min  swi_anom_min
count  74076.000000  74076.000000
mean       0.074450     -1.449189
std        0.162101      0.665563
min       -0.047764     -6.163972
25%       -0.005000     -1.769804
50%        0.014000     -1.380492
75%        0.073093     -1.044293
max        1.077539      2.805269
